# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ameen740/Internship_Flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

I will use the Week-5 baseline to create a ranked queue for human review. Pages with at least 100 search impressions and a CTR below 2.0% will be marked as CTR_FIX_CANDIDATE, while the remaining pages will receive NO_ACTION. The priority score will rank the candidates so reviewers can focus first on pages with meaningful search visibility and a potential CTR improvement opportunity. Each candidate will include a reason code and suggested action, making the recommendation easy to understand and review. This queue is intended as directional decision-support, not an automatic decision or a guarantee that changing a page will improve its performance.

In [10]:
# ML-10 - Load March 2026 Search Intelligence data

import duckdb
import pandas as pd
from google.colab import userdata

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_Token")

# Connect to DuckDB
con = duckdb.connect()

# Give DuckDB access to Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

# Hugging Face warehouse
warehouse = "hf://datasets/FlyRank/internship-warehouse"

# Load March 2026 data
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    gsc_data_available
FROM read_parquet(
    '{warehouse}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

df = con.sql(query).df()

print("Data loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Columns:", df.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data loaded successfully!
Rows: 9841378
Columns: 7
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_data_available']


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-10 — Q1: Ranked actions + reason codes

import pandas as pd
import numpy as np

# Use the Week-5 data
action_df = df.copy()

# Calculate CTR
action_df["ctr_pct"] = (
    100.0 * action_df["gsc_clicks"] /
    action_df["gsc_impressions"].replace(0, np.nan)
)

# Remove invalid values
action_df = action_df.replace(
    [np.inf, -np.inf],
    np.nan
)

action_df = action_df.dropna(
    subset=["gsc_impressions", "gsc_clicks", "ctr_pct"]
)

# Week-5 baseline rule
action_df["reason_code"] = np.where(
    (action_df["gsc_impressions"] >= 100) &
    (action_df["ctr_pct"] < 2.0),
    "CTR_FIX_CANDIDATE",
    "NO_ACTION"
)

# Priority score from the Week-5 baseline
action_df["priority_score"] = (
    action_df["gsc_impressions"] /
    (action_df["ctr_pct"] + 0.1)
)

# Suggested action for human review
action_df["suggested_action"] = np.where(
    action_df["reason_code"] == "CTR_FIX_CANDIDATE",
    "Review title, snippet, search intent, and page relevance",
    "Monitor; no immediate content action"
)

# Rank candidates first by reason code and then by priority
action_df = action_df.sort_values(
    ["reason_code", "priority_score"],
    ascending=[True, False]
).reset_index(drop=True)

action_df["rank"] = range(1, len(action_df) + 1)

# Create the review queue
queue = action_df[
    [
        "rank",
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr_pct",
        "priority_score",
        "reason_code",
        "suggested_action"
    ]
].copy()

# Display the top 20 candidates
top_20 = queue[
    queue["reason_code"] == "CTR_FIX_CANDIDATE"
].head(20)

print("Top 20 ranked review candidates:")
display(top_20)

print("\nReason-code counts:")
print(queue["reason_code"].value_counts())

print("\nTotal rows:", len(queue))
print(
    "CTR_FIX_CANDIDATE:",
    (queue["reason_code"] == "CTR_FIX_CANDIDATE").sum()
)
print(
    "NO_ACTION:",
    (queue["reason_code"] == "NO_ACTION").sum()
)

# Basic checks
assert queue["priority_score"].notna().all()
assert queue["reason_code"].isin(
    ["CTR_FIX_CANDIDATE", "NO_ACTION"]
).all()

print("\nAll Q1 checks passed.")

Top 20 ranked review candidates:


,rank,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr_pct,priority_score,reason_code,suggested_action
0,1,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.002495,391083.403758,CTR_FIX_CANDIDATE,"Review title, snippet, search intent, and page..."
1,2,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,0.000000,373680.000000,CTR_FIX_CANDIDATE,"Review title, snippet, search intent, and page..."
2,3,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,0.005128,371005.538375,CTR_FIX_CANDIDATE,"Review title, snippet, search intent, and page..."
3,4,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.000000,333830.000000,CTR_FIX_CANDIDATE,"Review title, snippet, search intent, and page..."
4,5,2026-03-27,client_23a62021009f63c4,content_44f34c0a90047651,32958,0,0.000000,329580.000000,CTR_FIX_CANDIDATE,"Review title, snippet, search intent, and page..."
5,6,2026-03-31,client_73cda7b4e4f265ea,content_fec55986a1868d62,31472,0,0.000000,314720.000000,CTR_FIX_CANDIDATE,"Review title, snippet, search intent, and page..."
6,7,2026-03-29,client_23a62021009f63c4,content_44f34c0a90047651,32756,2,0.006106,308710.880424,CTR_FIX_CANDIDATE,"Review title, snippet, search intent, and page..."
7,8,2026-03-25,client_23a62021009f63c4,content_44f34c0a90047651,30964,1,0.003230,299952.851958,CTR_FIX_CANDIDATE,"Review title, snippet, search intent, and page..."
8,9,2026-03-02,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28973,0,0.000000,289730.000000,CTR_FIX_CANDIDATE,"Review title, snippet, search intent, and page..."
9,10,2026-03-01,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28947,0,0.000000,289470.000000,CTR_FIX_CANDIDATE,"Review title, snippet, search intent, and page..."



Reason-code counts:
reason_code
NO_ACTION            2982116
CTR_FIX_CANDIDATE     628945
Name: count, dtype: int64

Total rows: 3611061
CTR_FIX_CANDIDATE: 628945
NO_ACTION: 2982116

All Q1 checks passed.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*
The ranked queue is intended for content and SEO reviewers who need a practical way to prioritize pages for manual review. It can be used to identify pages with meaningful search visibility and a potential CTR improvement opportunity, helping reviewers decide where to investigate first. The output is decision-support only: it does not prove that a page needs a content change or that a change will improve performance. The ranking is based on observed search signals from the available data and may be affected by seasonality, search demand, page intent, or other factors not captured by the model. Final content decisions should therefore remain with a human reviewer.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 - Question 2
# Intended use and limits check

intended_use = (
    "Prioritize pages for human review based on observed search visibility "
    "and CTR signals."
)

limits = [
    "Decision-support only; not an automatic content decision.",
    "Does not prove that a content change will improve performance.",
    "Results may be affected by seasonality and changes in search demand.",
    "Human review is required before taking action.",
]

print("Intended use:")
print(intended_use)

print("\nKnown limits:")
for limit in limits:
    print("-", limit)

# Basic checks
assert "human review" in intended_use.lower()
assert len(limits) >= 4

print("\nQ2 checks passed.")

Intended use:
Prioritize pages for human review based on observed search visibility and CTR signals.

Known limits:
- Decision-support only; not an automatic content decision.
- Does not prove that a content change will improve performance.
- Results may be affected by seasonality and changes in search demand.
- Human review is required before taking action.

Q2 checks passed.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A human reviewer should inspect the page context, search visibility, CTR, search intent, and the reason code before taking any content action. The reviewer should also consider whether the result could be explained by seasonality, changes in demand, or other factors that are not represented in the available features. The queue should only prioritize investigation; the reviewer makes the final decision. Content changes, publishing, deletion, pruning, or other irreversible actions should never be automated from this score alone. The system should also not automatically claim that a refresh will improve performance or that a ranking change was caused by a specific content action.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-10 - Question 3
# Human review rules and no-go list

human_review_rules = [
    "Check the page's search visibility and CTR before acting.",
    "Check whether the reason code matches the observed signals.",
    "Review search intent and page context.",
    "Consider seasonality, demand changes, and other possible explanations.",
    "Confirm the recommendation is appropriate before making a content change.",
]

no_go_list = [
    "Do not automatically publish content changes.",
    "Do not automatically delete or prune pages.",
    "Do not automatically rewrite titles or content.",
    "Do not automatically claim that a refresh will improve performance.",
    "Do not automatically claim that a content change caused a ranking or traffic recovery.",
]

print("Human review rules:")
for rule in human_review_rules:
    print("-", rule)

print("\nNo-go list:")
for item in no_go_list:
    print("-", item)

# Basic checks
assert len(human_review_rules) >= 5
assert len(no_go_list) >= 5

print("\nQ3 checks passed.")

Human review rules:
- Check the page's search visibility and CTR before acting.
- Check whether the reason code matches the observed signals.
- Review search intent and page context.
- Consider seasonality, demand changes, and other possible explanations.
- Confirm the recommendation is appropriate before making a content change.

No-go list:
- Do not automatically publish content changes.
- Do not automatically delete or prune pages.
- Do not automatically rewrite titles or content.
- Do not automatically claim that a refresh will improve performance.
- Do not automatically claim that a content change caused a ranking or traffic recovery.

Q3 checks passed.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

I would treat the recommendations as potentially stale if the underlying search signals change noticeably or if the ranked queue starts behaving differently from the validation results. I would monitor impressions, clicks, CTR, the share of pages receiving each reason code, missing-value rates, and the measured precision of the highest-priority review queue. A sustained shift in these signals, a drop in review precision, or a material change in the data distribution would trigger investigation. I would consider revalidation or retraining when these changes persist or when new labeled outcomes become available. These triggers are intended to flag when the decision-support playbook may need review, not to automatically retrain or deploy a new model.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-10 - Q4: Monitoring / retrain triggers

monitoring_triggers = [
    "Sustained change in impressions, clicks, or CTR distributions",
    "Material change in the share of each reason code",
    "Increase in missing or invalid feature values",
    "Sustained drop in measured precision of the top review queue",
    "New labeled outcomes become available",
]

print("Monitoring / retrain triggers:")

for trigger in monitoring_triggers:
    print("-", trigger)

assert len(monitoring_triggers) == 5

print("\nQ4 checks passed.")

Monitoring / retrain triggers:
- Sustained change in impressions, clicks, or CTR distributions
- Material change in the share of each reason code
- Increase in missing or invalid feature values
- Sustained drop in measured precision of the top review queue
- New labeled outcomes become available

Q4 checks passed.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked review queue will be exported to work/outputs/ so it can be reused as an input to the research paper. The exported file contains the ranking, observed search signals, priority score, reason code, and suggested human-review action. The queue is regenerated by the notebook rather than treated as a permanent production dataset. Any figures selected for reuse will be saved separately under work/figures/. These exports are intended to preserve the measured outputs and make the paper's recommendations traceable to the notebook.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-10 - Q5: Exports for the paper

from pathlib import Path

# Create the required output directory
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export the ranked queue generated in Question 1
queue_path = output_dir / "ranked_action_queue.csv"

queue.to_csv(queue_path, index=False)

print("Queue exported successfully!")
print("File:", queue_path)
print("Rows:", len(queue))
print("Columns:", len(queue.columns))

# Verify that the export exists
assert queue_path.exists()

# Verify that the exported file can be read back
exported_queue = pd.read_csv(queue_path)

assert len(exported_queue) == len(queue)
assert list(exported_queue.columns) == list(queue.columns)

print("\nExport verification passed.")


Queue exported successfully!
File: work/outputs/ranked_action_queue.csv
Rows: 3611061
Columns: 10

Export verification passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.